<img align="left" src="img/ECE364-logo.png" width="300px" style="padding:30px;border:thin solid white;"> 

# Lecture 21 - Words and Attention (aka the basic transformer model) 
## ECE364 - Programming Methods for Machine Learning
### Nickvash Kani 












###### Slides based off prior lectures by Alex Schwing, Aigou Han, Farzad Kamalabadi, Corey Snyder. All mistakes are my own!

In this lecture: 

- Discuss the basic structure of the trasnformer model
- Discuss embeddings

## Where did we leave language processing? 

When we last talked about generating language, it was with recurrent neural networks: 

<img align="center" src="img/rnn-unfolded.png" width="800px" style="padding:30px;border:thin solid white;"> 


But there are problems with simple RNNs: 

- **Slow** - RNNs process data sequentially making them super slow. 
- **Vanishing gradient** -  RNNs (including LSTMs and GRUs) struggle with learning long range dependencies due to vanishing gradients. [1]  
- **Lack of attention** - RNNs process each word in order, and have difficulty with non-sequential dependencies. 

## Attention is all you need

In 2017, Ashish Vaswani and his colleagues at Google Research/Brain published their seminal work "Attention is all you need" [2]. In it, they introduce the transformer architecture which has become the foundation of every large language model since. So in this lecture, let's go through and try to decipher this architecture: 

<img align="center" src="img/Attention_is_all_you_need.png" width="400px" style="padding:30px;border:thin solid white;"> 


<img align="left" src="img/Attention_is_all_you_need_tokenization.png" width="300px" style="padding:30px;border:thin solid white;"> 


## Step 1: Tokenization

We brushed on tokenization previously, but the basic idea is that you can segment language by letters, words, or something in between. What features you use can significantly impact your model's performance and structure.

<img align="center" src="img/tokenization_schemes.png" width="800px" style="padding:30px;border:thin solid white;"> 


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load a tiny GPT-2 model (small enough for CPU)
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# Make sure model is in eval mode
model.eval()

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

text = "The quick brown fox"
input_ids = tokenizer.encode(text)
tokens = tokenizer.tokenize(text)
words = [tokenizer.decode([id]) for id in input_ids]

print("Token IDs: ")
print(input_ids)
print("Tokens: ")
print(tokens)
print("Decoded tokens: ")
print(words)
print(f"Number of tokens: {len(tokens)}")

| Concept | Explanation |
|:---|:---|
| Token $\neq$ Word | A word may be split into multiple tokens if it’s rare/complex. |
| Special Space Tokens | GPT-2 uses a `Ġ` (special marker) to represent leading spaces in tokens. |
| Number of tokens | Depends on how text is broken into subwords. |
| In this case | `"The quick brown fox"` tokenizes to

## Inference

How would we generate a totally new sequence from the transformer? 

First let's look at what we get when we insert a input sequence into the model:

In [ ]:
input_ids = tokenizer.encode("The quick brown fox", return_tensors='pt').to('cpu')
print(input_ids)

# Forward pass through model
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits  # (batch_size, seq_length, vocab_size)
    
print(logits.shape)
print(logits)

## What do the logits represent?

You can think of the logit matrix will be of the form: `[batch_size, sequence_length, vocab_size]`

Each slice `logits[:, i, :]` (for some $i$) represents the model’s prediction of what token should come next after the $i$-th token. 

Specifically:

- `logits[:, 0, :]` → predict what comes after the **first token**
- `logits[:, 1, :]` → predict what comes after the **second token**
- `logits[:, 2, :]` → predict what comes after the **third token**
- ...
- `logits[:, -1, :]` → predict what comes after the **very last token** you input

That's why when generating the next token, you only care about the last position — `logits[:, -1, :]`.

It’s predicting the next token based on the full context so far.


How do we generate a long sequence?

<img align="center" src="img/inference_diagram.png" width="800px" style="padding:30px;border:thin solid white;"> 

So let's generate a long sequence. 

In [ ]:
def generate_step(model, tokenizer, input_text, device='cpu'):
    """
    Given the current input_text, run one generation step.
    - Return updated text
    - Return top 10 next-token candidates (token, probability)
    """
    # Tokenize input
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(device)

    # Forward pass through model
    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits  # (batch_size, seq_length, vocab_size)

    # Focus on the last token's logits
    next_token_logits = logits[:, -1, :].squeeze(0)  # (vocab_size,)

    # Convert logits to probabilities
    probs = torch.softmax(next_token_logits, dim=-1)

    # Get top 10 most probable tokens
    top_probs, top_indices = torch.topk(probs, 10)

    # Decode top tokens
    top_tokens = [tokenizer.decode(idx.item()) for idx in top_indices]

    # Greedy choice: pick the token with the highest probability
    next_token_id = top_indices[0].unsqueeze(0)
    next_token = tokenizer.decode(next_token_id)

    # Append the new token to the existing text
    updated_text = input_text + next_token

    # Prepare top 10 as a list of (token, probability) pairs
    top_10 = [(token, prob.item()) for token, prob in zip(top_tokens, top_probs)]

    return updated_text, top_10

In [ ]:
# Initial prompt
text = "The meaning of life is"

# Generation steps
for _ in range(5):
    text, top_10 = generate_step(model, tokenizer, text)
    print("Top 10 next tokens:")
    for token, prob in top_10:
        print(f"{token!r} ({prob:.4f})")
    print(f"Updated Text: {repr(text)}\n")
    print("\n" + "-"*50 + "\n")


Any guesses what the meaning of life is to DistilGPT2?

In [ ]:
def generate_n_tokens(model, tokenizer, input_text, n_tokens=5, device='cpu', verbose=True):
    """
    Generate `n_tokens` tokens from input_text using greedy decoding.
    - verbose=True prints generation steps
    - returns the final updated text
    """
    text = input_text

    for i in range(n_tokens):
        text, top_10 = generate_step(model, tokenizer, text, device=device)

        if verbose:
            print(f"Step {i+1}:")
            print(f"Updated Text: {repr(text)}\n")
            print("Top 10 next tokens:")
            for token, prob in top_10:
                print(f"{token!r} ({prob:.4f})")
            print("\n" + "-"*50 + "\n")

    return text

In [ ]:
start_text = "The meaning of life is to"
final_text = generate_n_tokens(model, tokenizer, start_text, n_tokens=20, verbose=False)
print("Final generated text:\n", final_text)

## Embeddings

Recall the PCA lecture where we extracted the principal components of the MNIST dataset and plotted the different MNIST images in a two-dimensional plot: 

<img align="center" src="img/MNIST_PCA_plot.png" width="800px" style="padding:30px;border:thin solid white;"> 

We are effectively storing information about the images as a two-dimensional vector. This is called **embedding**. We as embedded the images as a two-dimensional vector. 

Consider the encoder-decoder scheme of the image segmentation network: 

<img align="center" src="img/autoencoder-example.png" width="800px" style="padding:30px;border:thin solid white;"> 

In the encoder part of the network, we go from a large image to small matrix. That small matrix contains the semantic information describing the image. We effectively **embed** the image as that small matrix. 


### Embedding words

Embedding a word is pretty much the same idea. The natwork wants to store that symbol as a vector and the direction of the vector encodes some semantic information about the word. 

<img align="center" src="img/analogy_visual.png" width="800px" style="padding:30px;border:thin solid white;"> 

What's interesting is that if the embedding is done well, the relative position between embeddings is itself a embedding and contains some semantic information! We can see this is code by analyzing one of the OG embedding schemes GLoVe embeddings: 

In [ ]:
import gensim.downloader
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load the GloVe 50-dimensional model
model = gensim.downloader.load("glove-wiki-gigaword-50")

# 1. Basic embedding lookup
word = "tower"
vector = model[word]
print(f"Embedding for '{word}':\n{vector}\n")

# 2. Embedding math: analogy task (king - man + woman ≈ queen)
result_vector = model["king"] - model["man"] + model["woman"]

# Find closest word to result_vector
similar_words = model.similar_by_vector(result_vector, topn=5)
print("Closest words to 'king - man + woman':")
for word, similarity in similar_words:
    print(f"  {word}: {similarity:.3f}")

print()

# 3. Cosine similarity between two words
def cosine_sim(w1, w2):
    v1 = model[w1].reshape(1, -1)
    v2 = model[w2].reshape(1, -1)
    return cosine_similarity(v1, v2)[0][0]

sim = cosine_sim("king", "queen")
print(f"Cosine similarity between 'king' and 'queen': {sim:.3f}")

sim2 = cosine_sim("tower", "building")
print(f"Cosine similarity between 'tower' and 'building': {sim2:.3f}")

print()

# 4. Create a custom vector (mix two concepts) and find nearby words
custom_vector = (model["river"] + model["mountain"]) / 2
similar_words = model.similar_by_vector(custom_vector, topn=5)
print("Words similar to average of 'river' and 'mountain':")
for word, similarity in similar_words:
    print(f"  {word}: {similarity:.3f}")

The important thing to remember is that computer accepts numbers, not symbols. So let's assume word tokenization. We need to embed the words as n-dimensional vectors. 

<img align="center" src="img/input_embedding.png" width="800px" style="padding:30px;border:thin solid white;"> 

* remember, the embedding matrix is trainable. Just another set of parameters that needs to be train of thousands of iterations on large datasets

<img align="left" src="img/Attention_is_all_you_need_positional_encoding.png" width="300px" style="padding:30px;border:thin solid white;"> 


## Step 2: Positional encoding

**Core concept:** We want to give the model a method to reference a word at a particular positon in the sentence.

**Some nuances:** 

- We want each word to carry information about its position in the sentence
- We want words that are close together to have similar positional encodings and the words that are far apart to have dissimilar position encodings. 
- Needs to be something that the model can learn. 
- Would be nice to only encode the encodings once (so no variable encodings for each sentence) 

<img align="left" src="img/positional_encoding.png" width="800px" style="padding:30px;border:thin solid white;"> 


### Sinusoidal Positional Encoding - (Vaswani et al., 2017)

<img align="right" src="img/positional_encoding_2.png" width="600px" style="padding:30px;border:thin solid white;"> 



In the original Transformer paper ("Attention is All You Need"), **fixed sinusoidal functions** were used:

For a position $pos$ and dimension $i$, the encoding is defined as:

$$
\text{PE}^0_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

$$
\text{PE}^1_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

Where:
- $pos$ = position in the sequence
- $i$ = embedding dimension index
- $d_{\text{model}}$ = total embedding size (e.g., 512)

**Even dimensions** → sine  
**Odd dimensions** → cosine

### Intuition:
- Different frequencies allow each position to be uniquely encoded.
- Nearby positions have similar encodings (smooth changes).
- Generalizes to sequences **longer** than what the model was trained on.

### Conceptual explanation

I found a [blog post](https://kazemnejad.com/blog/transformer_architecture_positional_encoding/) that gives the best explanation so far about why the positonal encodings are formulated the way they are. Here's the cliff notes: 

We need a positional encoding that satisfies the realtive criteria: 
- It should output a unique encoding for each time-step (word’s position in a sentence)
- Distance between any two time-steps should be consistent across sentences with different lengths.
- Our model should generalize to longer sentences without any efforts. Its values should be bounded.
- It must be deterministic.


#### Why sines and cosines? 

Suppose we want to represent a number in binary format: 

|   | Bit 3 | Bit 2 | Bit 1 | Bit 0 |   | Bit 3 | Bit 2 | Bit 1 | Bit 0 |
|:-:|:----:|:----:|:----:|:----:|:-:|:----:|:----:|:----:|:----:|
|  0 |  0 | 0 | 0 | 0 | 8 | 1 | 0 | 0 | 0 |
|  1 |  0 | 0 | 0 | 1 | 9 | 1 | 0 | 0 | 1 |
|  2 |  0 | 0 | 1 | 0 | 10 | 1 | 0 | 1 | 0 |
|  3 |  0 | 0 | 1 | 1 | 11 | 1 | 0 | 1 | 1 |
|  4 |  0 | 1 | 0 | 0 | 12 | 1 | 1 | 0 | 0 |
|  5 |  0 | 1 | 0 | 1 | 13 | 1 | 1 | 0 | 1 |
|  6 |  0 | 1 | 1 | 0 | 14 | 1 | 1 | 1 | 0 |
|  7 |  0 | 1 | 1 | 1 | 15 | 1 | 1 | 1 | 1 |

Each bit position has a different frequency. That is pretty much what we're doing with the positonal vectors above representng the dimension and pos as frequencies. 

#### Relative positioning

The other benefit is that we can calculate relative postioning fairly easily:

$$
M \cdot 
\begin{bmatrix}
\sin(\omega_k t) \\
\cos(\omega_k t)
\end{bmatrix}
=
\begin{bmatrix}
\sin(\omega_k (t + \phi)) \\
\cos(\omega_k (t + \phi))
\end{bmatrix}
$$

---

**Proof:**

Let $M$ be a $2 \times 2$ matrix, we want to find $u_1, v_1, u_2$ and $v_2$ so that:

$$
\begin{bmatrix}
u_1 & v_1 \\
u_2 & v_2
\end{bmatrix}
\cdot
\begin{bmatrix}
\sin(\omega_k t) \\
\cos(\omega_k t)
\end{bmatrix}
=
\begin{bmatrix}
\sin(\omega_k (t + \phi)) \\
\cos(\omega_k (t + \phi))
\end{bmatrix}
$$

By applying the [addition theorem](https://en.wikipedia.org/wiki/Trigonometric_identities#Angle_sum_and_difference_identities), we can expand the right-hand side as follows:

$$
\begin{bmatrix}
u_1 & v_1 \\
u_2 & v_2
\end{bmatrix}
\cdot
\begin{bmatrix}
\sin(\omega_k t) \\
\cos(\omega_k t)
\end{bmatrix}
=
\begin{bmatrix}
\sin(\omega_k t) \cos(\omega_k \phi) + \cos(\omega_k t) \sin(\omega_k \phi) \\
\cos(\omega_k t) \cos(\omega_k \phi) - \sin(\omega_k t) \sin(\omega_k \phi)
\end{bmatrix}
$$

Which results in the following two equations:

$$
u_1 \sin(\omega_k t) + v_1 \cos(\omega_k t) = \cos(\omega_k \phi) \sin(\omega_k t) + \sin(\omega_k \phi) \cos(\omega_k t) \tag{1}
$$

$$
u_2 \sin(\omega_k t) + v_2 \cos(\omega_k t) = -\sin(\omega_k \phi) \sin(\omega_k t) + \cos(\omega_k \phi) \cos(\omega_k t) \tag{2}
$$

By solving the above equations, we get:

$$
u_1 = \cos(\omega_k \phi) \quad v_1 = \sin(\omega_k \phi)
$$

$$
u_2 = -\sin(\omega_k \phi) \quad v_2 = \cos(\omega_k \phi)
$$

---

Thus, the final transformation matrix $M$ is:

$$
M_{\phi,k} =
\begin{bmatrix}
\cos(\omega_k \phi) & \sin(\omega_k \phi) \\
-\sin(\omega_k \phi) & \cos(\omega_k \phi)
\end{bmatrix}
$$

**This is likely why sine/cosine are both used**. There isn't a clear linear transformation with only sine or only cosine.  

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_positional_encoding(seq_len=100, d_model=64, save_path=None):
    """
    Plots the positional encoding and optionally saves the figure.
    
    Args:
        seq_len (int): Number of positions (x-axis).
        d_model (int): Embedding depth (y-axis).
        save_path (str, optional): If provided, saves the figure to this path.
    """
    def get_sinusoidal_encoding(seq_len, d_model):
        pos = np.arange(seq_len)[:, None]
        i = np.arange(d_model)[None, :]
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / d_model)
        angle_rads = pos * angle_rates

        pos_encoding = np.zeros(angle_rads.shape)
        pos_encoding[:, 0::2] = np.sin(angle_rads[:, 0::2])
        pos_encoding[:, 1::2] = np.cos(angle_rads[:, 1::2])

        return pos_encoding

    pos_encoding = get_sinusoidal_encoding(seq_len, d_model)

    plt.figure(figsize=(8, 6))
    plt.imshow(pos_encoding.T, aspect='auto', cmap='viridis', origin='lower')
    plt.colorbar(label='Encoding Value')
    plt.xlabel('Position')
    plt.ylabel('Depth (Embedding Dimension)')
    plt.title('Positional Encoding (Sinusoidal)')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"Figure saved to {save_path}")
    else:
        plt.show()

# Example usage:
plot_positional_encoding(seq_len=100, d_model=64, save_path=None)  # Show figure
# plot_positional_encoding(seq_len=100, d_model=64, save_path='positional_encoding.png')  # Save figure

### Why is Positional Encoding a Function of Sin/Cos?

When designing the Transformer (Vaswani et al., 2017), the key challenge was:

> Transformers have no recurrence and no convolution — how do we tell the model the order of tokens?

#### Core Reasons for Using Sinusoids

1. Captures Relative and Absolute Position

- Sinusoids are smooth and periodic.
- They encode both:
  - **Absolute position**: Each position gets a unique vector.
  - **Relative distance**: Easy to infer how far apart two positions are.

Note: 
$$
\sin(a + b) = \sin(a)\cos(b) + \cos(a)\sin(b)
$$
Thus, the model can **easily compute relative offsets** based on the sin/cos values.

2. No Extra Parameters

- Sinusoidal functions are **fixed** — no extra weights to learn.
- Transformer can **extrapolate** to longer sequences during inference without retraining.

3. Multi-Scale Representations

- Different dimensions have different frequencies:
  - Low-frequency sinusoids capture **long-range** patterns.
  - High-frequency sinusoids capture **local** patterns.

> "We use sine and cosine functions of different frequencies. For each position, we generate a vector whose even indices are sine functions and whose odd indices are cosine functions of different wavelengths."

This lets the model attend both to **nearby** and **distant** tokens effectively.

---

#### There are other position encoding schemes [5]: 

> "There are many choices of positional encodings, learned and fixed ... We also experimented with using learned positional embeddings instead, and found that the two versions produced nearly identical results"

## That's it for today

- Next time we'll discuss attention
- HW10 (last HW!) will be released tomorrow
- And most importantly, have a good weekend.

## References

[1] Bengio, Yoshua, Patrice Simard, and Paolo Frasconi. "Learning long-term dependencies with gradient descent is difficult." IEEE transactions on neural networks 5.2 (1994): 157-166.

[2] Vaswani, Ashish, et al. "Attention is all you need." Advances in neural information processing systems 30 (2017).

[3] Umar Jamil "Attention is all you need (Transformer) - Model explanation (including math), Inference and Training" - https://www.youtube.com/watch?v=bCz4OMemCcA

[4] 3Blue1Brown "Attention in transformers, step-by-step | DL6" https://www.youtube.com/watch?v=eMlx5fFNoYc

[5] Adrian Tam, "Positional Encodings in Transformer Models" https://machinelearningmastery.com/positional-encodings-in-transformer-models/